# Week 3: Deep Learning with PyTorch

## Learning Objectives
- Understand PyTorch tensors and autograd
- Build networks with nn.Module
- Use DataLoaders for mini-batch training
- Train and evaluate a PyTorch model

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    print(f'PyTorch version: {torch.__version__}')
    TORCH = True
except ImportError:
    print('PyTorch not installed: pip install torch')
    TORCH = False

## 1. Tensors and Autograd

In [ ]:
if TORCH:
    x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
    y = x ** 2 + 3 * x + 1
    y.sum().backward()
    print('Gradient (2x+3):', x.grad)
else:
    print('PyTorch not available.')

## 2. nn.Module

In [ ]:
if TORCH:
    class BinaryClassifier(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(64, 1), nn.Sigmoid()
            )
        def forward(self, x): return self.net(x)

    from sklearn.datasets import load_breast_cancer
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    import numpy as np

    cancer = load_breast_cancer()
    X, y = cancer.data, cancer.target
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_tr_t = torch.FloatTensor(scaler.fit_transform(X_train))
    y_tr_t = torch.FloatTensor(y_train)
    X_te_t = torch.FloatTensor(scaler.transform(X_test))
    y_te_t = torch.FloatTensor(y_test)

    model = BinaryClassifier(X.shape[1])
    print(f'Params: {sum(p.numel() for p in model.parameters()):,}')
else:
    print('PyTorch not available.')

In [ ]:
if TORCH:
    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()
    losses = []
    for epoch in range(50):
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb).squeeze(), yb)
            loss.backward(); optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))

    model.eval()
    with torch.no_grad():
        acc = ((model(X_te_t).squeeze() > 0.5) == y_te_t).float().mean()
    print(f'Test Accuracy: {acc:.4f}')

    import matplotlib.pyplot as plt
    plt.plot(losses); plt.title('Training Loss'); plt.xlabel('Epoch')
    plt.tight_layout(); plt.show()
else:
    print('PyTorch not available.')

## Exercise

1. Add a learning rate scheduler to the training loop
2. Implement early stopping manually
3. Save and reload the model with `torch.save` and `torch.load`

In [ ]:
# Your code here


## Summary

- ✓ PyTorch tensors and autograd
- ✓ nn.Module and nn.Sequential
- ✓ DataLoader for mini-batch training

## Next Week
Convolutional Neural Networks (CNNs)!